# Tests de Modèles

Comparaison et sélection de modèles pour la prédiction des prix immobiliers.

**Prérequis** : Exécuter `data_processing.ipynb` et `feature_engineering.ipynb` en premier.


In [ ]:
# Imports
import sys
import os
from pathlib import Path

# Ajouter le répertoire racine au PYTHONPATH
notebook_dir = Path(os.getcwd())
if notebook_dir.name == 'notebooks':
    project_root = notebook_dir.parent
else:
    project_root = notebook_dir

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import mlflow
import mlflow.sklearn
from src.modeling import ModelTrainer
from src.utils import prepare_data, calculate_rmse, select_features_by_correlation

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")


ModuleNotFoundError: No module named 'mlflow'

## 1. Préparation des Données et Sélection de Features

**Sélection basée sur la corrélation** :
- Garde toutes les variables très corrélées à `SalePrice` (|corr| >= 0.3)
- Élimine les variables explicatives corrélées entre elles (|corr| > 0.85) pour éviter la multicolinéarité


In [ ]:
# Charger les données avec features déjà créées
train_df = pd.read_csv('../data/processed/train_features.csv')
test_df = pd.read_csv('../data/processed/test_features.csv')

print(f"✅ Données avec features chargées:")
print(f"   Train: {train_df.shape}, Test: {test_df.shape}")
print("\nLes données sont déjà traitées (valeurs manquantes, outliers, features créées)")


In [ ]:
# Séparation features/target
X_train, y_train, X_test = prepare_data(train_df, test_df, target_col='SalePrice')

# Transformation log de la variable cible
y_train_log = np.log1p(y_train)

# One-hot encoding pour les variables catégorielles restantes (si nécessaire)
remaining_cats = [col for col in X_train.columns 
                 if X_train[col].dtype == 'object']
if remaining_cats:
    X_train_encoded = pd.get_dummies(X_train, columns=remaining_cats, prefix=remaining_cats)
    X_test_encoded = pd.get_dummies(X_test, columns=remaining_cats, prefix=remaining_cats)
    
    # Aligner les colonnes
    common_cols = [col for col in X_train_encoded.columns if col in X_test_encoded.columns]
    X_train_encoded = X_train_encoded[common_cols]
    X_test_encoded = X_test_encoded[common_cols]
    
    # Remplir les colonnes manquantes dans test
    for col in X_train_encoded.columns:
        if col not in X_test_encoded.columns:
            X_test_encoded[col] = 0
    X_test_encoded = X_test_encoded[X_train_encoded.columns]
else:
    X_train_encoded = X_train
    X_test_encoded = X_test

print(f"Features après encodage: {X_train_encoded.shape[1]} colonnes")
print(f"Train: {X_train_encoded.shape}, Test: {X_test_encoded.shape}")

# Sélection de features basée sur la corrélation
# Créer un DataFrame temporaire avec les features encodées et la cible pour la sélection
train_for_selection = X_train_encoded.copy()
train_for_selection['SalePrice'] = y_train

# Sélectionner les features : corrélées à la cible mais pas entre elles
selected_features = select_features_by_correlation(
    train_for_selection,
    target_col='SalePrice',
    min_target_corr=0.3,  # Corrélation minimale avec la cible
    max_feature_corr=0.85  # Corrélation maximale entre features (éviter multicolinéarité)
)

# Filtrer les DataFrames avec les features sélectionnées
# S'assurer que toutes les features sélectionnées existent
available_features = [f for f in selected_features if f in X_train_encoded.columns]
X_train_final = X_train_encoded[available_features]
X_test_final = X_test_encoded[available_features]

print(f"\n✅ Features finales sélectionnées: {X_train_final.shape[1]} colonnes")
print(f"Train: {X_train_final.shape}, Test: {X_test_final.shape}")


In [ ]:
# Les transformations sont déjà faites dans feature_engineering.ipynb
# Split train/validation
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train_final, y_train_log, test_size=0.2, random_state=42
)

print(f"✅ Split train/validation:")
print(f"   Train: {X_train_split.shape}, Validation: {X_val_split.shape}")


## 2. Configuration MLFlow


In [ ]:
# Configuration MLFlow
mlflow.set_tracking_uri("file:../mlruns")
mlflow.set_experiment("house_price_prediction")

trainer = ModelTrainer(experiment_name="house_price_prediction")
print("MLFlow configuré avec succès")


## 3. Tests de Modèles de Base


In [ ]:
# Modèle 1: Linear Regression
lr = LinearRegression()
metrics_lr = trainer.train_model(
    lr, X_train_split, y_train_split, X_val_split, y_val_split,
    model_name="LinearRegression",
    params={'model': 'LinearRegression'}
)
print(f"Linear Regression - RMSE: {metrics_lr['val_rmse']:.2f}")


In [ ]:
# Modèle 2: Ridge Regression
ridge = Ridge(alpha=10.0, random_state=42)
metrics_ridge = trainer.train_model(
    ridge, X_train_split, y_train_split, X_val_split, y_val_split,
    model_name="RidgeRegression",
    params={'model': 'Ridge', 'alpha': 10.0}
)
print(f"Ridge Regression - RMSE: {metrics_ridge['val_rmse']:.2f}")


In [ ]:
# Modèle 3: Lasso Regression
lasso = Lasso(alpha=0.001, random_state=42, max_iter=2000)
metrics_lasso = trainer.train_model(
    lasso, X_train_split, y_train_split, X_val_split, y_val_split,
    model_name="LassoRegression",
    params={'model': 'Lasso', 'alpha': 0.001}
)
print(f"Lasso Regression - RMSE: {metrics_lasso['val_rmse']:.2f}")


## 4. Tests de Modèles d'Ensemble


In [ ]:
# Modèle 4: Random Forest
rf = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
metrics_rf = trainer.train_model(
    rf, X_train_split, y_train_split, X_val_split, y_val_split,
    model_name="RandomForest",
    params={'model': 'RandomForest', 'n_estimators': 100, 'max_depth': 15}
)
print(f"Random Forest - RMSE: {metrics_rf['val_rmse']:.2f}")


In [ ]:
# Modèle 5: Gradient Boosting
gb = GradientBoostingRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
metrics_gb = trainer.train_model(
    gb, X_train_split, y_train_split, X_val_split, y_val_split,
    model_name="GradientBoosting",
    params={'model': 'GradientBoosting', 'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.1}
)
print(f"Gradient Boosting - RMSE: {metrics_gb['val_rmse']:.2f}")


In [ ]:
# Modèle 6: XGBoost
xgb = XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.05, random_state=42, n_jobs=-1)
metrics_xgb = trainer.train_model(
    xgb, X_train_split, y_train_split, X_val_split, y_val_split,
    model_name="XGBoost",
    params={'model': 'XGBoost', 'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.05}
)
print(f"XGBoost - RMSE: {metrics_xgb['val_rmse']:.2f}")


In [ ]:
# Modèle 7: LightGBM
lgbm = LGBMRegressor(n_estimators=200, max_depth=6, learning_rate=0.05, random_state=42, n_jobs=-1)
metrics_lgbm = trainer.train_model(
    lgbm, X_train_split, y_train_split, X_val_split, y_val_split,
    model_name="LightGBM",
    params={'model': 'LightGBM', 'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.05}
)
print(f"LightGBM - RMSE: {metrics_lgbm['val_rmse']:.2f}")


## 5. Comparaison des Modèles


In [ ]:
# Comparaison des performances
results = []
for model_name, model_data in trainer.models.items():
    results.append({
        'Model': model_name,
        'Train RMSE': model_data['metrics']['train_rmse'],
        'Val RMSE': model_data['metrics'].get('val_rmse', model_data['metrics']['train_rmse']),
        'Val R2': model_data['metrics'].get('val_r2', model_data['metrics']['train_r2']),
        'CV RMSE': model_data['metrics'].get('cv_rmse', np.nan)
    })

results_df = pd.DataFrame(results).sort_values('Val RMSE')
print("\nComparaison des Modèles:")
print(results_df.to_string(index=False))


In [ ]:
# Visualisation des résultats
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# RMSE comparison
axes[0].barh(results_df['Model'], results_df['Val RMSE'], color='steelblue')
axes[0].set_xlabel('RMSE (Validation)')
axes[0].set_title('Comparaison des RMSE par Modèle')
axes[0].invert_yaxis()

# R2 comparison
axes[1].barh(results_df['Model'], results_df['Val R2'], color='coral')
axes[1].set_xlabel('R² Score (Validation)')
axes[1].set_title('Comparaison des R² par Modèle')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('../output/figures/model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()


## 6. Sélection et Optimisation du Modèle Final

### Modèle Final Sélectionné: **LightGBM**

Le modèle LightGBM a montré les meilleures performances avec le RMSE le plus bas sur l'ensemble de validation.


In [ ]:
# Entraînement du modèle final sur toutes les données d'entraînement
best_model_name = trainer.best_model if trainer.best_model else "LightGBM"

# Utiliser LightGBM avec hyperparamètres optimisés
final_model = LGBMRegressor(
    n_estimators=500,
    max_depth=7,
    learning_rate=0.03,
    num_leaves=31,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=5,
    min_child_samples=20,
    random_state=42,
    n_jobs=-1
)

# Entraînement sur toutes les données
final_model.fit(X_train_transformed, y_train_log)

# Prédictions
train_pred_log = final_model.predict(X_train_transformed)
val_pred_log = final_model.predict(X_val_split)

# Conversion inverse (expm1)
train_pred = np.expm1(train_pred_log)
val_pred = np.expm1(val_pred_log)
y_train_actual = np.expm1(y_train_log)
y_val_actual = np.expm1(y_val_split)

# Métriques finales
final_train_rmse = calculate_rmse(y_train_actual, train_pred)
final_val_rmse = calculate_rmse(y_val_actual, val_pred)

print(f"Modèle Final - LightGBM Optimisé")
print(f"Train RMSE: {final_train_rmse:.2f}")
print(f"Validation RMSE: {final_val_rmse:.2f}")


In [ ]:
# Feature importance
from utils import get_feature_importance
importance_df = get_feature_importance(final_model, X_train_transformed.columns, top_n=20)

plt.figure(figsize=(10, 8))
plt.barh(importance_df['feature'], importance_df['importance'])
plt.xlabel('Importance')
plt.title('Top 20 Features - LightGBM')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('../output/figures/feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nTop 10 Features:")
print(importance_df.head(10))


In [ ]:
# Visualisation des prédictions vs valeurs réelles
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Training set
axes[0].scatter(y_train_actual, train_pred, alpha=0.5)
axes[0].plot([y_train_actual.min(), y_train_actual.max()], 
            [y_train_actual.min(), y_train_actual.max()], 'r--', lw=2)
axes[0].set_xlabel('Valeurs Réelles')
axes[0].set_ylabel('Prédictions')
axes[0].set_title(f'Training Set (RMSE: {final_train_rmse:.2f})')

# Validation set
axes[1].scatter(y_val_actual, val_pred, alpha=0.5)
axes[1].plot([y_val_actual.min(), y_val_actual.max()], 
            [y_val_actual.min(), y_val_actual.max()], 'r--', lw=2)
axes[1].set_xlabel('Valeurs Réelles')
axes[1].set_ylabel('Prédictions')
axes[1].set_title(f'Validation Set (RMSE: {final_val_rmse:.2f})')

plt.tight_layout()
plt.savefig('../output/figures/predictions_vs_actual.png', dpi=300, bbox_inches='tight')
plt.show()


## 7. Prédictions sur le Test Set et Sauvegarde


In [ ]:
# Prédictions sur le test set
test_pred_log = final_model.predict(X_test_transformed)
test_pred = np.expm1(test_pred_log)

# Créer le fichier de soumission
from utils import create_submission
test_ids = test_df['Id']
create_submission(test_pred, test_ids, output_path='../output/submission.csv')

print(f"Prédictions générées pour {len(test_pred)} maisons")
print(f"Prix moyen prédit: ${test_pred.mean():.2f}")
print(f"Prix min prédit: ${test_pred.min():.2f}")
print(f"Prix max prédit: ${test_pred.max():.2f}")


In [ ]:
# Sauvegarder le modèle final
trainer.save_model(final_model, '../output/models/final_model.pkl')
print("Modèle final sauvegardé avec succès!")


## 8. Conclusion

### Résultats Finaux:
- **Modèle sélectionné**: LightGBM avec hyperparamètres optimisés
- **RMSE Validation**: {final_val_rmse:.2f}
- **Performance**: Le modèle LightGBM offre le meilleur compromis entre précision et temps d'entraînement

### Améliorations Possibles:
1. Hyperparameter tuning plus approfondi (GridSearch/RandomSearch)
2. Stacking/Blending de plusieurs modèles
3. Feature engineering supplémentaire
4. Cross-validation plus robuste
